<a href="https://colab.research.google.com/github/GoudoMahan/AI-agent-practice/blob/main/HM_5_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Week 5 Task 1：Prompt Engineering · SFT · DPO

---

姓名：胡豪达

学号：523021910471

---


本作业分为三个部分：

| 部分 | 内容 |
|------|------|
| Part 1 | Prompt Engineering（反向设计）|
| Part 2 | SFT（监督微调）|
| Part 3 | DPO（偏好学习）|

**⚠️ 重要提示：**
- 请确保选择 **GPU 运行时**（`运行时 → 更改运行时类型 → T4 GPU`）
- 所有标有 `# ✏️ 学生填写` 的单元格都需要你完成

---
## 🔧 环境安装

运行以下单元格安装所需依赖（大约需要 2-3 分钟）。

In [1]:
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.2 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 4 packages in 176ms
Prepared 1 package in 112ms
Uninstalled 1 package in 2ms
Installed 1 package in 7ms
 - trl==0.24.0
 + trl==0.22.2
Using Python 3.12.13 environment at: /usr
Resolved 28 packages in 268ms
Prepared 1 package in 728ms
Uninstalled 1 package in 110ms
Installed 1 package in 79ms
 - transformers==5.3.0
 + transformers==5.2.0
Using Python 3.12.13 environment at: /usr
Resolved 55 packages in 234ms
Prepared 4 packages in 19.59s
Installed 4 packages in 51ms
 + causal-conv1d==1.6.0
 + fla-core==0.4.2
 + flash-linear-attention==0.4.2
 + ninja==1.13.0


---
## 🚀 加载模型

我们使用 Unsloth 加速加载 Qwen3.5-2B 模型，并配置 LoRA 适配器以支持后续微调。

In [2]:
import unsloth  # noqa: F401 — 先于本单元格内任何其他 ML 库
from unsloth import FastVisionModel
import torch

MODEL_NAME = "unsloth/Qwen3.5-2B"

def load_model_with_lora():
    """(重新)加载原始模型并挂载 LoRA（纯文本任务：不微调视觉层）"""
    model, tokenizer = FastVisionModel.from_pretrained(
        MODEL_NAME,
        load_in_4bit = False,
        use_gradient_checkpointing = "unsloth",
    )
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers     = False,
        finetune_language_layers   = True,
        finetune_attention_modules = True,
        finetune_mlp_modules       = True,
        r = 16,
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )
    return model, tokenizer

model, tokenizer = load_model_with_lora()
print("✅ 模型加载完成！")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

processor_config.json: 0.00B [00:00, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Unsloth: Making `model.base_model.model.model.language_model` require gradients
✅ 模型加载完成！


### 定义推理辅助函数

以下函数用于在整个作业中调用模型生成回答。

In [3]:
import torch
from transformers import TextStreamer

def _msg_text(text: str):
    """Qwen3.5-VL 风格：content 为块列表（本作业仅 text，不传真实图像）。"""
    return [{"type": "text", "text": text}]

def generate_response(prompt, max_tokens=512, temperature=0.7, print_output=True):
    """调用模型生成回答（FastVisionModel：tokenizer 走图像占位 + 文本）"""
    FastVisionModel.for_inference(model)

    messages = [
        {"role": "user", "content": _msg_text(prompt)}
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        None,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    if inputs.get("attention_mask") is None:
        inputs["attention_mask"] = torch.ones_like(inputs["input_ids"], dtype=torch.long)

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id

    gen_kwargs = dict(
        max_new_tokens=max_tokens,
        use_cache=True,
        temperature=temperature,
        min_p=0.1,
        pad_token_id=pad_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    if print_output:
        streamer = TextStreamer(tokenizer, skip_prompt=True)
        outputs = model.generate(**inputs, streamer=streamer, **gen_kwargs)
    else:
        outputs = model.generate(**inputs, **gen_kwargs)

    in_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][in_len:], skip_special_tokens=True)
    return response

print("✅ 推理函数已定义")

✅ 推理函数已定义


In [4]:
test_response = generate_response("请用一句话介绍你自己。") #首次运行时可能需要等待几分钟
print("\n--- 模型加载验证完成 ---")

我是 Qwen3.5，由阿里巴巴最新推出的通义千问大语言模型，具备强大的语言理解、逻辑推理及多模态分析能力，可协助您完成从简单问答到复杂任务的全方位支持。<|im_end|>

--- 模型加载验证完成 ---


---
---

# 🧪 Part 1 — Prompt Engineering（反向设计）（30 分）

## 🎯 目标

你将**只看到模型输出结果（Answer）**，而看不到原始的 prompt。

你的任务是：编写一个 prompt，使模型对**同样的问题**能生成与给定输出尽可能接近的结果。


---
## 🧾 任务 1 — 结构化任务规划（10 分）

### 🔹 输入问题

> 用户要从厨房拿牛奶，然后去客厅看电视。

### 🔹 给定输出（你需要设计 prompt 让模型产生接近的输出）

```json
{
  "reasoning": "该任务涉及家庭环境中功能区域之间的顺序导航，需要先取回物品再前往目标区域执行活动。",
  "plan": [
    {"step": 1, "action": "navigate", "target": "厨房", "purpose": "取牛奶"},
    {"step": 2, "action": "pick_up", "target": "牛奶"},
    {"step": 3, "action": "navigate", "target": "客厅", "purpose": "看电视"},
    {"step": 4, "action": "operate", "target": "电视", "operation": "turn_on"}
  ]
}
```

### 🎯 你的任务

设计一个 prompt，使模型对**同一个问题**生成与上述输出尽可能接近的结果。注意观察输出的格式、风格、字段结构等特征。

In [5]:
# ✏️ 学生填写：设计你的 prompt
# 提示：你的 prompt 应包含指令和用户输入，使模型输出与给定输出相同风格的结果

task1_prompt = """
Role：你是一个智能家庭助手的任务规划核心。你的任务是接收用户的自然语言指令，并将其解析为结构化的 JSON 格式的任务列表。

Goal：分析用户指令中的意图，拆解为逻辑顺序的步骤，输出包含推理过程和具体行动计划的 JSON 对象。

Constraints：
1. 输出必须是纯粹的 JSON 格式，不要包含 markdown 代码块标记（如 ```json ... ```）或任何额外的解释性文字。
2. `reasoning` 字段需简要分析任务的逻辑和意图。
3. `plan` 字段是一个列表，按时间顺序包含每一步操作。
4. 每个步骤必须包含：`step` (序号), `action` (动作类型), `target` (目标对象/地点), `purpose` 或 `operation` (目的或具体操作细节)。

Input Example：
用户要从厨房拿牛奶，然后去客厅看电视。

Output Example：
{
  "reasoning": "该任务涉及家庭环境中功能区域之间的顺序导航，需要先取回物品再前往目标区域执行活动。",
  "plan": [
    {"step": 1, "action": "navigate", "target": "厨房", "purpose": "取牛奶"},
    {"step": 2, "action": "pick_up", "target": "牛奶"},
    {"step": 3, "action": "navigate", "target": "客厅", "purpose": "看电视"},
    {"step": 4, "action": "operate", "target": "电视", "operation": "turn_on"}
  ]
}

User Instruction：
用户要从厨房拿牛奶，然后去客厅看电视。
"""

print("=" * 50)
print("任务 1 — 你的 Prompt 输出：")
print("=" * 50)
task1_output = generate_response(task1_prompt)

任务 1 — 你的 Prompt 输出：
{
  "reasoning": "该任务涉及家庭环境中功能区域之间的顺序导航，需要先取回物品再前往目标区域执行活动。",
  "plan": [
    {
      "step": 1,
      "action": "navigate",
      "target": "厨房",
      "purpose": "取牛奶"
    },
    {
      "step": 2,
      "action": "pick_up",
      "target": "牛奶",
      "purpose": "获取所需物品"
    },
    {
      "step": 3,
      "action": "navigate",
      "target": "客厅",
      "purpose": "看电视"
    },
    {
      "step": 4,
      "action": "operate",
      "target": "电视",
      "operation": "turn_on"
    }
  ]
}<|im_end|>


---
## 🧾 任务 2 — 风格与表达控制（10 分）

### 🔹 输入问题

> 解释 Transformer 模型的基本原理。

### 🔹 给定输出

```
话说这 Transformer 啊，那可是深度学习界的一大宝贝！
它不走寻常路，全靠「注意力」三个字吃饭。
怎么回事呢？就好比你看一篇文章，哪个词重要、哪个词不重要，它门儿清！
用行话讲，这叫 Attention(Q,K,V) = softmax(QK^T / √d_k)V
甭管多长的句子，它照样玩得转！
```

### 🎯 你的任务

设计一个 prompt，使模型对**同一个问题**生成与上述输出尽可能接近的结果。注意观察输出的语言风格（口语化说书风格）、句式特征、内容组成等。

In [13]:
# ✏️ 学生填写：设计你的 prompt

task2_prompt = """
Role：你是一位精通深度学习的“北京通”，擅长用最接地气、最幽默的“京味儿”大白话，把高深的 AI 技术讲得连胡同口的大爷都能听懂。

Style Guidelines：
1. 语气：必须像聊天一样，使用口语化词汇（如“宝贝”、“门儿清”、“玩得转”、“怎么着”）。
2. 句式：多用短句、反问句和感叹句，避免教科书式的陈述。
3. 比喻：用生活中的例子（如看文章、找重点）来类比技术概念（如自注意力机制）。

Input Example：
解释 Transformer 模型的基本原理。

Output Example：
话说这 Transformer 啊，那可是深度学习界的一大宝贝！
它不走寻常路，全靠「注意力」三个字吃饭。
怎么回事呢？就好比你看一篇文章，哪个词重要、哪个词不重要，它门儿清！
用行话讲，这叫 Attention(Q,K,V) = softmax(QK^T / √d_k)V
甭管多长的句子，它照样玩得转！

User Instruction：请解释 Transformer 模型的基本原理。
"""

print("=" * 50)
print("任务 2 — 你的 Prompt 输出：")
print("=" * 50)
task2_output = generate_response(task2_prompt)

任务 2 — 你的 Prompt 输出：
话说这 Transformer 啊，那可是深度学习界的一大宝贝！

它不走寻常路，全靠「注意力」三个字吃饭。

怎么回事呢？就好比你看一篇文章，哪个词重要、哪个词不重要，它门儿清！
用行话讲，这叫 Attention(Q,K,V) = softmax(QK^T / √d_k)V
甭管多长的句子，它照样玩得转！

**核心就俩字：把重点找出来！**

以前咱们学神经网络，就像给文章贴标签，从头到尾扫一遍，谁重要谁不重要，全凭运气，或者靠死记硬背。

Transformer 不一样，它有个超能力：**自注意力机制（Self-Attention）**。

这就好比你在胡同口看热闹，突然有人喊：“哎，你瞅瞅那谁！他刚才多说了两句，肯定是大事！”
Transformer 立马就能锁定那个“谁”，不管文章多长，多乱，它都能瞬间算出：
*   谁跟谁关系最铁？
*   谁才是主角？
*   谁只是背景板？

它把句子拆成一个个小方块（Token），然后像雷达一样，疯狂地互相“对对碰”。
*   如果 A 和 B 挨得近，它们就“撞”出火花，说明关系紧密；
*   如果 C 和 D 离得远，它们就“撞”不出火花，说明关系疏远。

最后，它把这些“火花”加权算个总分，给每个词打分。
**得分高的，就是重点；得分低的，就是陪衬。**

这就好比你在写论文，不用从头到尾念一遍，你直接挑出最关键的结论，再配上最相关的论据，剩下的废话全扔一边。

所以啊，Transformer 就是给机器装了一个“超级大脑”，让它能像人一样，**一眼看穿**整篇文章的脉络，精准地抓住重点，再把它变成机器也能听懂的语言。

怎么样，这逻辑是不是比看说明书还顺溜？<|im_end|>


---
## 🧾 任务 3 — 错误修复与增强（10 分）

### 🔹 输入

```json
{
  "task": "去厨房拿苹果"
  "steps": ["去厨房", "拿苹果"]
}
```

### 🔹 给定输出

```json
{
  "task": "前往厨房取回苹果",
  "steps": ["导航至厨房", "拾取苹果"],
  "fix_explanation": "修复了字段间缺失的逗号，规范了任务描述用词，并提升了步骤表述的清晰度。"
}
```

### 🎯 你的任务

设计一个 prompt，使模型对**同一个输入**生成与上述输出尽可能接近的结果。注意观察模型做了哪些修改、增加了什么字段、fix_explanation 的内容等。

In [15]:
# ✏️ 学生填写：设计你的 prompt

task3_prompt = """
Role：
你是一个专业的JSON数据清洗与标准化助手。你的任务是接收包含语法错误或表述不规范的JSON输入，修复其语法错误，并优化其内容表述，使其符合标准、清晰且一致的风格。

Goal：
1.  修复语法错误：确保输出的JSON格式完全正确，可以被标准解析器解析。
2.  标准化表述：将任务（`task`）和步骤（`steps`）的描述从口语化、简单的形式，优化为更正式、清晰、动词导向的指令性语言。
3.  提供修复说明：简要解释你所做的关键修改。

Constraints：
1.  输出必须是严格的JSON格式。
2.  输出必须包含三个字段：`task` (字符串), `steps` (字符串数组), `fix_explanation` (字符串)。
3.  `steps` 数组中的每个步骤都应是一个清晰、可执行的原子操作。
4.  `fix_explanation` 字段应简明扼要地概括主要修复和优化点。

Input Example：
{
  "task": "去厨房拿苹果",
  "steps": ["去厨房", "拿苹果"]
}

Output Example：
{
  "task": "前往厨房取回苹果",
  "steps": ["导航至厨房", "拾取苹果"],
  "fix_explanation": "修复了字段间缺失的逗号，规范了任务描述用词，并提升了步骤表述的清晰度。"
}

User Instruction:
{
  "task": "去厨房拿苹果"
  "steps": ["去厨房", "拿苹果"]
}
"""

print("=" * 50)
print("任务 3 — 你的 Prompt 输出：")
print("=" * 50)
task3_output = generate_response(task3_prompt)

任务 3 — 你的 Prompt 输出：
{
  "task": "前往厨房取回苹果",
  "steps": [
    "导航至厨房",
    "拾取苹果"
  ],
  "fix_explanation": "修复了输入中字段间缺失的逗号，规范了任务描述用词，并提升了步骤表述的清晰度。"
}<|im_end|>


---
---

# 🔧 Part 2 — SFT 监督微调（35 分）

## 🎯 目标

通过构造训练数据，使模型学会：
- 稳定输出结构化结果（JSON）
- 执行多步推理
- 对齐任务格式


---
## 🧾 Step 1：训练数据构造（15 分）

你需要自行构造 **≥ 50 条**训练数据。

### 📌 数据格式

每条数据为一个 `instruction` + `output` 的对：

```python
{"instruction": "...", "output": "..."}
```

### 📌 数据要求

数据必须覆盖以下类型：

1. **结构化任务规划**：自然语言指令 → JSON 格式的行动计划
2. **多步推理**：任务规划 → step-by-step 输出
3. **格式严格约束**：输出必须为合法 JSON
4. **反例修复**：给定错误示范 → 输出正确答案

### 示例

```python
# 示例 1：任务规划
{"instruction": "用户要去厨房拿水杯，然后回书房工作。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
 "output": '{"reasoning": "该任务需要依次从当前位置前往厨房取回水杯，再前往书房进行工作。", "plan": [{"step": 1, "action": "navigate", "target": "厨房", "purpose": "取水杯"}, {"step": 2, "action": "pick_up", "target": "水杯"}, {"step": 3, "action": "navigate", "target": "书房", "purpose": "工作"}, {"step": 4, "action": "sit_down", "target": "书桌"}]}'}

# 示例 2：格式修复
{"instruction": "修复以下错误JSON并增强描述：{\"task\": \"去浴室洗手\" \"steps\": [\"去浴室\", \"洗\"]}",
 "output": '{"task": "前往浴室清洗双手", "steps": ["导航至浴室", "用肥皂洗手"], "fix_explanation": "修复了字段间缺失的逗号，规范了任务描述，并提升了步骤描述的具体性。"}'}
```

In [18]:
# ✏️ 学生填写：构造你的训练数据（≥ 50 条）
# 请确保数据覆盖上述四种类型

sft_training_data = [
    # === 类型 1：任务规划 → 结构化 JSON ===
    {
        "instruction": "用户要去厨房拿水杯，然后回书房工作。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
        "output": '{"reasoning": "该任务需要依次从当前位置前往厨房取回水杯，再前往书房进行工作。", "plan": [{"step": 1, "action": "navigate", "target": "厨房", "purpose": "取水杯"}, {"step": 2, "action": "pick_up", "target": "水杯"}, {"step": 3, "action": "navigate", "target": "书房", "purpose": "工作"}, {"step": 4, "action": "sit_down", "target": "书桌"}]}'
    },
    {
        "instruction": "用户要去厨房拿水杯，然后回书房工作。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
        "output": {
            "reasoning": "该任务需要依次从当前位置前往厨房取回水杯，再前往书房进行工作。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "厨房", "purpose": "取水杯"},
                {"step": 2, "action": "pick_up", "target": "水杯"},
                {"step": 3, "action": "navigate", "target": "书房", "purpose": "工作"},
                {"step": 4, "action": "sit_down", "target": "书桌"}
            ]
        }
    },
    {
        "instruction": "用户需要去客厅打开电视，然后调高音量。请生成包含 reasoning 和 plan 的 JSON 计划。",
        "output": {
            "reasoning": "任务需先导航至客厅执行设备操作，再调整设备参数以满足需求。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "客厅", "purpose": "操作电视"},
                {"step": 2, "action": "operate", "target": "电视", "operation": "turn_on"},
                {"step": 3, "action": "adjust", "target": "音量", "value": "调高"}
            ]
        }
    },
    {
        "instruction": "请规划用户去卧室换睡衣后，到客厅看电视的 JSON 行动计划。",
        "output": {
            "reasoning": "该任务涉及更换衣物和娱乐活动，需先完成更衣再导航至目标区域使用设备。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "卧室", "purpose": "换睡衣"},
                {"step": 2, "action": "change", "target": "睡衣", "purpose": "更换着装"},
                {"step": 3, "action": "navigate", "target": "客厅", "purpose": "看电视"},
                {"step": 4, "action": "operate", "target": "电视", "operation": "turn_on"}
            ]
        }
    },
    {
        "instruction": "用户要去阳台给植物浇水，然后回书房继续写作。生成对应的 JSON 计划。",
        "output": {
            "reasoning": "任务包含园艺维护和创作活动，需先完成照料植物再切换至工作场景。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "阳台", "purpose": "浇水植物"},
                {"step": 2, "action": "water", "target": "植物", "tool": "喷壶"},
                {"step": 3, "action": "navigate", "target": "书房", "purpose": "继续写作"},
                {"step": 4, "action": "continue", "target": "写作任务"}
            ]
        }
    },
    {
        "instruction": "请输出用户从玄关拿钥匙，然后出门的 JSON 格式行动计划。",
        "output": {
            "reasoning": "该任务为外出准备流程，需先获取必要物品再执行离开动作。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "玄关", "purpose": "取钥匙"},
                {"step": 2, "action": "pick_up", "target": "钥匙"},
                {"step": 3, "action": "open", "target": "大门", "purpose": "出门准备"},
                {"step": 4, "action": "exit", "target": "家门"}
            ]
        }
    },
    {
        "instruction": "用户需去厨房热牛奶，然后端到餐厅喝。请生成结构化 JSON 计划。",
        "output": {
            "reasoning": "任务包含食物加热和用餐流程，需先处理食材再转移至用餐区域。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "厨房", "purpose": "热牛奶"},
                {"step": 2, "action": "heat", "target": "牛奶", "tool": "微波炉"},
                {"step": 3, "action": "carry", "target": "热牛奶", "to": "餐厅"},
                {"step": 4, "action": "drink", "target": "牛奶"}
            ]
        }
    },
    {
        "instruction": "请规划用户去健身房锻炼，锻炼后去浴室洗澡的 JSON 行动计划。",
        "output": {
            "reasoning": "任务涉及健身和清洁流程，需按顺序完成锻炼再进行个人卫生处理。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "健身房", "purpose": "锻炼"},
                {"step": 2, "action": "exercise", "target": "身体", "duration": "30分钟"},
                {"step": 3, "action": "navigate", "target": "浴室", "purpose": "洗澡"},
                {"step": 4, "action": "take", "target": "淋浴", "purpose": "清洁身体"}
            ]
        }
    },
    {
        "instruction": "用户需要从书房拿笔记本，然后去会议室开会。输出 JSON 格式的计划。",
        "output": {
            "reasoning": "该任务为工作场景转移，需先获取工具再导航至会议地点执行协作任务。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "书房", "purpose": "取笔记本"},
                {"step": 2, "action": "pick_up", "target": "笔记本"},
                {"step": 3, "action": "navigate", "target": "会议室", "purpose": "参加开会"},
                {"step": 4, "action": "join", "target": "会议", "purpose": "协作讨论"}
            ]
        }
    },
    {
        "instruction": "请生成用户去客厅关窗，然后调整空调温度的行动计划（JSON格式）。",
        "output": {
            "reasoning": "任务涉及环境控制，需先完成物理操作再调整电子设备参数以维持舒适环境。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "客厅", "purpose": "关窗"},
                {"step": 2, "action": "close", "target": "窗户"},
                {"step": 3, "action": "navigate", "target": "空调控制面板", "purpose": "调温度"},
                {"step": 4, "action": "adjust", "target": "温度", "value": "调至舒适范围"}
            ]
        }
    },
    {
        "instruction": "用户要去卧室拿手机，然后回客厅回复消息。请输出包含 reasoning 的 JSON 计划。",
        "output": {
            "reasoning": "任务需先获取通讯工具，再转移至合适区域执行信息处理任务。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "卧室", "purpose": "取手机"},
                {"step": 2, "action": "pick_up", "target": "手机"},
                {"step": 3, "action": "navigate", "target": "客厅", "purpose": "回复消息"},
                {"step": 4, "action": "respond", "target": "消息", "tool": "手机"}
            ]
        }
    },
    {
        "instruction": "请规划用户去厨房切水果，然后端到阳台吃的行动计划（JSON格式）。",
        "output": {
            "reasoning": "任务包含食材处理和休闲用餐，需先完成备餐再转移至休闲区域享用。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "厨房", "purpose": "切水果"},
                {"step": 2, "action": "cut", "target": "水果", "tool": "刀具"},
                {"step": 3, "action": "carry", "target": "水果盘", "to": "阳台"},
                {"step": 4, "action": "enjoy", "target": "水果", "location": "阳台"}
            ]
        }
    },
    {
        "instruction": "用户需去书房打印文件，然后去会议室分发。生成对应的 JSON 行动计划。",
        "output": {
            "reasoning": "该任务为文档处理与协作流程，需先完成文件输出再转移至会议场景执行分发动作。",
            "plan": [
                {"step": 1, "action": "navigate", "target": "书房", "purpose": "打印文件"},
                {"step": 2, "action": "print", "target": "文件", "tool": "打印机"},
                {"step": 3, "action": "navigate", "target": "会议室", "purpose": "分发文件"},
                {"step": 4, "action": "distribute", "target": "打印文件", "to": "参会人员"}
            ]
        }
    },

    # === 类型 2：多步推理任务 ===
    {
        "instruction": "用户要去厨房拿水杯，然后回书房工作。请输出多步推理的行动计划。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "厨房",
                    "reasoning": "用户当前位置可能在书房或客厅，需先确定水杯位置（通常在厨房），因此需导航至厨房以获取必要物品。"
                },
                {
                    "step": 2,
                    "action": "pick_up",
                    "target": "水杯",
                    "reasoning": "到达厨房后，需明确水杯的具体位置（如橱柜或台面），通过视觉识别并执行抓取动作，确保携带所需物品。"
                },
                {
                    "step": 3,
                    "action": "navigate",
                    "target": "书房",
                    "reasoning": "取水完成后，用户核心需求是工作，书房通常配备工作所需设施（如书桌、电脑），因此需返回该区域以继续任务。"
                },
                {
                    "step": 4,
                    "action": "sit_down",
                    "target": "书桌前",
                    "reasoning": "进入书房后，坐下可减少体力消耗并提升专注度，为后续工作（如使用电脑或书写）创造稳定环境。"
                }
            ]
        }
    },
    {
        "instruction": "用户需去客厅打开电视，然后调高音量。请生成包含多步推理的 JSON 计划。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "客厅",
                    "reasoning": "电视通常位于客厅，用户需先到达该区域才能执行设备操作，确保物理位置正确。"
                },
                {
                    "step": 2,
                    "action": "locate",
                    "target": "电视遥控器",
                    "reasoning": "控制电视需使用遥控器，可能放置在茶几或电视柜上，需通过视觉搜索定位以执行下一步操作。"
                },
                {
                    "step": 3,
                    "action": "press_button",
                    "target": "电源键",
                    "reasoning": "确认遥控器后，按下电源键可启动电视，满足用户观看需求的基础条件。"
                },
                {
                    "step": 4,
                    "action": "adjust",
                    "target": "音量+键",
                    "reasoning": "用户可能因环境噪音或初始音量过低而需调高音量，通过遥控器按键提升听觉体验。"
                }
            ]
        }
    },
    {
        "instruction": "请规划用户去卧室换睡衣后，到客厅看电视的 JSON 行动计划（含推理）。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "卧室",
                    "reasoning": "睡衣通常存放在卧室（如衣柜或抽屉），需先到达该区域以获取更换衣物，满足舒适需求。"
                },
                {
                    "step": 2,
                    "action": "retrieve",
                    "target": "睡衣",
                    "reasoning": "进入卧室后，需从存储位置取出睡衣，确保衣物可及性以执行更换动作。"
                },
                {
                    "step": 3,
                    "action": "change_clothes",
                    "target": "睡衣",
                    "reasoning": "更换睡衣可提升居家舒适度，为后续休闲活动（如看电视）创造放松状态。"
                },
                {
                    "step": 4,
                    "action": "navigate",
                    "target": "客厅",
                    "reasoning": "完成更衣后，用户需求转向娱乐，客厅配备电视等设施，需转移至该区域以满足观看需求。"
                }
            ]
        }
    },
    {
        "instruction": "用户要去阳台给植物浇水，然后回书房继续写作。生成对应的推理式行动计划。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "阳台",
                    "reasoning": "植物通常放置在阳台以获得光照，用户需到达该区域执行养护任务，确保植物健康。"
                },
                {
                    "step": 2,
                    "action": "retrieve",
                    "target": "喷壶",
                    "reasoning": "浇水需使用工具（如喷壶或水桶），提前获取可减少中途中断，提高效率。"
                },
                {
                    "step": 3,
                    "action": "water_plants",
                    "target": "所有植物",
                    "reasoning": "逐一对植物浇水可满足其生长需求，避免遗漏导致植物枯萎，维护居家环境美观。"
                },
                {
                    "step": 4,
                    "action": "navigate",
                    "target": "书房",
                    "reasoning": "完成园艺任务后，用户核心工作未完成，书房具备写作所需条件（如安静环境与书写工具），需返回继续任务。"
                }
            ]
        }
    },
    {
        "instruction": "请输出用户从玄关拿钥匙，然后出门的推理式 JSON 计划。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "玄关",
                    "reasoning": "钥匙通常存放于玄关处（如钥匙盒或挂钩），需到达该区域以获取外出必需品。"
                },
                {
                    "step": 2,
                    "action": "identify",
                    "target": "正确钥匙",
                    "reasoning": "若有多把钥匙，需通过视觉或触觉识别目标钥匙（如对应门锁的钥匙），避免取错影响外出计划。"
                },
                {
                    "step": 3,
                    "action": "pick_up",
                    "target": "钥匙",
                    "reasoning": "确认钥匙后，抓取并携带至门口，为后续开门动作做准备。"
                },
                {
                    "step": 4,
                    "action": "open",
                    "target": "大门",
                    "reasoning": "使用钥匙解锁大门，执行旋转或推动动作，完成外出前的最后物理障碍。"
                }
            ]
        }
    },
    {
        "instruction": "用户需去厨房热牛奶，然后端到餐厅喝。请生成多步推理的结构化计划。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "厨房",
                    "reasoning": "加热牛奶需在厨房使用加热设备（如微波炉或炉灶），确定功能区域以满足处理需求。"
                },
                {
                    "step": 2,
                    "action": "locate",
                    "target": "牛奶容器",
                    "reasoning": "需找到存放牛奶的位置（如冰箱或橱柜），确保食材可获取以进行下一步处理。"
                },
                {
                    "step": 3,
                    "action": "heat",
                    "target": "牛奶",
                    "tool": "微波炉",
                    "reasoning": "微波炉加热快速且便捷，适合液态食物加热，设置合理时间（如1-2分钟）以避免过热。"
                },
                {
                    "step": 4,
                    "action": "carry",
                    "target": "热牛奶",
                    "to": "餐厅",
                    "reasoning": "餐厅通常配备餐桌和餐具，更适合用餐环境，转移牛奶可提升饮用体验。"
                }
            ]
        }
    },
    {
        "instruction": "用户计划去健身房锻炼，锻炼后去浴室洗澡。请输出含推理的 step-by-step 计划。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "健身房",
                    "reasoning": "健身房配备锻炼设备（如跑步机或哑铃），需到达该区域以执行健身计划，满足健康或体能目标。"
                },
                {
                    "step": 2,
                    "action": "exercise",
                    "target": "指定项目（如跑步）",
                    "duration": "30分钟",
                    "reasoning": "设定明确锻炼内容与时长可量化目标，避免过度或不足，确保训练效果。"
                },
                {
                    "step": 3,
                    "action": "navigate",
                    "target": "浴室",
                    "reasoning": "锻炼后身体出汗，浴室具备清洁设施（如淋浴），需转移至该区域以满足个人卫生需求。"
                },
                {
                    "step": 4,
                    "action": "take",
                    "target": "淋浴",
                    "reasoning": "淋浴可快速清洁身体并放松肌肉，为锻炼后的恢复提供必要步骤。"
                }
            ]
        }
    },
    {
        "instruction": "用户要从书房拿笔记本，然后去会议室开会。请生成包含多步推理的 JSON 计划。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "书房",
                    "reasoning": "笔记本通常存放于书房工作区域，需到达该位置以获取会议所需工具，确保信息记录能力。"
                },
                {
                    "step": 2,
                    "action": "identify",
                    "target": "正确笔记本",
                    "reasoning": "若有多台设备，需确认会议指定的笔记本（如含特定资料的电脑），避免携带错误设备影响协作。"
                },
                {
                    "step": 3,
                    "action": "power_on",
                    "target": "笔记本",
                    "reasoning": "提前开机可缩短启动时间，确保进入会议室时设备已就绪，提升会议效率。"
                },
                {
                    "step": 4,
                    "action": "navigate",
                    "target": "会议室",
                    "reasoning": "会议室配备投影和座椅，适合多人协作，携带设备到达该区域可满足正式会议需求。"
                }
            ]
        }
    },
    {
        "instruction": "用户需去客厅关窗，然后调整空调温度。请生成推理式 step-by-step 行动计划。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "客厅",
                    "reasoning": "窗户和空调通常位于同一区域（客厅或卧室），需先到达客厅以执行环境控制任务。"
                },
                {
                    "step": 2,
                    "action": "close",
                    "target": "所有窗户",
                    "reasoning": "关闭窗户可减少室外噪音、灰尘或温度影响，为空调调节创造封闭环境，提升效率。"
                },
                {
                    "step": 3,
                    "action": "navigate",
                    "target": "空调控制面板",
                    "reasoning": "空调面板用于温度设置，可能位于墙面或独立遥控器，需定位以执行参数调整。"
                },
                {
                    "step": 4,
                    "action": "set_temperature",
                    "target": "24℃",
                    "reasoning": "24℃通常为舒适温度范围，结合关窗后的环境，可维持室内恒温并节能。"
                }
            ]
        }
    },
    {
        "instruction": "用户要去卧室拿手机，然后回客厅回复消息。请输出含推理的结构化计划。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "卧室",
                    "reasoning": "手机可能充电或存放于卧室（如床头柜），需到达该区域以获取通讯工具，满足消息处理需求。"
                },
                {
                    "step": 2,
                    "action": "unlock",
                    "target": "手机",
                    "reasoning": "解锁可访问消息应用，确保隐私安全的同时允许查看和回复信息。"
                },
                {
                    "step": 3,
                    "action": "navigate",
                    "target": "客厅",
                    "reasoning": "客厅空间更宽敞，可能配备沙发等舒适设施，适合较长时间的回复任务或协作讨论。"
                },
                {
                    "step": 4,
                    "action": "respond",
                    "target": "消息",
                    "tool": "手机",
                    "reasoning": "使用手机在客厅回复消息，可结合舒适体位和环境，提升处理效率与体验。"
                }
            ]
        }
    },
    {
        "instruction": "请规划用户去厨房切水果，然后端到阳台吃的行动计划（含多步推理）。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "厨房",
                    "reasoning": "切水果需使用刀具和案板，厨房配备必要工具与清洁设施，确保操作安全与卫生。"
                },
                {
                    "step": 2,
                    "action": "prepare",
                    "target": "水果与工具",
                    "reasoning": "取出水果（如苹果、西瓜）并清洗刀具，减少交叉污染风险，保障食品安全。"
                },
                {
                    "step": 3,
                    "action": "cut",
                    "target": "水果",
                    "tool": "刀具",
                    "reasoning": "将水果切块可提升食用便捷性，适合直接取用或搭配餐具，增强食用体验。"
                },
                {
                    "step": 4,
                    "action": "carry",
                    "target": "果盘",
                    "to": "阳台",
                    "reasoning": "阳台通常具备自然光和休闲空间，端水果至此可享受户外环境，提升用餐心情。"
                }
            ]
        }
    },
    {
        "instruction": "用户需要整理书桌文件，然后打印重要文档。请生成多步推理的 JSON 计划。",
        "output": {
            "plan": [
                {
                    "step": 1,
                    "action": "navigate",
                    "target": "书房书桌",
                    "reasoning": "文件通常存放于工作区域，需到达书桌整理资料，确保打印前内容准确且有序。"
                },
                {
                    "step": 2,
                    "action": "sort",
                    "target": "文件",
                    "category": "按项目分类",
                    "reasoning": "分类文件可减少打印错误或遗漏，例如区分合同、报告等不同文档类型。"
                },
                {
                    "step": 3,
                    "action": "identify",
                    "target": "需打印文档",
                    "reasoning": "从分类文件中筛选出紧急或必要资料，避免冗余打印，节约资源。"
                },
                {
                    "step": 4,
                    "action": "navigate",
                    "target": "打印机位置",
                    "reasoning": "打印机可能位于书房或公共区域，需确定其位置以执行打印任务。"
                },
                {
                    "step": 5,
                    "action": "print",
                    "target": "选定文档",
                    "reasoning": "通过打印机输出纸质版，满足存档、签字或分享需求，确保信息可传递。"
                }
            ]
        }
    },
    # === 类型 3：格式严格约束 ===
    {
        "instruction": "修复以下错误json：{\"task\": \"去浴室洗手\" \"steps\": [\"去浴室\", \"洗\"]}",
        "output": {
            "task": "前往浴室清洗双手",
            "steps": ["导航至浴室", "用肥皂洗手"],
            "fix_explanation": "修复字段间缺失逗号，规范任务描述并细化步骤。"
        }
    },
    {
        "instruction": "修复以下错误json：{'task': '整理书桌', 'steps': ['清理杂物', '摆放书籍']}",
        "output": {
            "task": "整理书桌并归类物品",
            "steps": ["清除桌面杂物", "分类摆放书籍与文件"],
            "fix_explanation": "替换单引号为双引号，拆分步骤并补充分类细节。"
        }
    },
    {
        "instruction": "修复以下错误json：{\"tasK\": \"做饭\" \"steps\": [洗米, 煮饭, 炒菜]}",
        "output": {
            "task": "完成烹饪流程",
            "steps": ["淘洗大米并煮饭", "准备食材并烹饪菜肴", "出锅装盘"],
            "fix_explanation": "修正键名大小写，为步骤添加双引号并补充完整流程。"
        }
    },
    {
        "instruction": "修复以下错误json：{\"task\": \"清洁房间\" \"steps\": [扫地 拖地]}",
        "output": {
            "task": "全面清洁房间",
            "steps": ["清扫地面垃圾", "拖地并消毒污渍区域"],
            "fix_explanation": "添加步骤间逗号，拆分动作并明确清洁工具。"
        }
    },
    {
        "instruction": "修复以下错误json：{\"task\": \"洗衣服\" \"steps\": [放衣服; 启动洗衣机]}",
        "output": {
            "task": "操作洗衣机洗衣",
            "steps": ["分类放入衣物", "添加洗涤剂并启动程序"],
            "fix_explanation": "替换分号为逗号，修正步骤描述并补充洗涤剂细节。"
        }
    },
    {
        "instruction": "修复以下错误json：{\"task\": \"购物\" \"steps\": [去超市, 结账离开, ]}",
        "output": {
            "task": "完成超市采购",
            "steps": ["选取商品", "收银台结账并核对清单"],
            "fix_explanation": "移除末尾多余逗号，细化步骤至可执行动作。"
        }
    },
    {
        "instruction": "修复以下错误json：{\"task\": \"锻炼\" \"steps\": {跑步: 30分钟, 健身: 20分钟}}",
        "output": {
            "task": "执行健身计划",
            "steps": ["户外跑步30分钟", "力量训练及拉伸20分钟"],
            "fix_explanation": "将对象转换为数组，统一时间格式并明确运动类型。"
        }
    },
    {
        "instruction": "修复以下错误json：{\"task\": \"学习\" \"steps\": [看书 做笔记 复习]}",
        "output": {
            "task": "系统性学习",
            "steps": ["阅读教材并标记重点", "整理笔记与思维导图", "复习并巩固知识点"],
            "fix_explanation": "添加步骤分隔逗号，拆分学习流程至逻辑闭环。"
        }
    },
    {
        "instruction": "修复以下错误json：{\"task\": \"准备会议\" \"steps\": [收集资料, 写报告打印文件]}",
        "output": {
            "task": "筹备会议材料",
            "steps": ["整理会议资料", "撰写报告并审核内容", "打印文件及检查设备"],
            "fix_explanation": "补充缺失逗号，拆分打印步骤并添加设备检查项。"
        }
    },
    {
        "instruction": "修复以下错误json：{\"task\": \"浇花\" \"steps\": [取水, 浇水至满, 晒太阳\"}]}",
        "output": {
            "task": "科学养护植物",
            "steps": ["取水并调节水温", "适量浇水至土壤湿润", "移至散射光处养护"],
            "fix_explanation": "修正引号闭合错误，补充水量控制与光照要求。"
        }
    },
    {
        "instruction": "修复以下错误json：{\"task\": \"组装家具\" \"steps\": [阅读说明书安装零件]}",
        "output": {
            "task": "完成家具组装",
            "steps": ["核对零件清单", "按说明书步骤组装并检查稳固性"],
            "fix_explanation": "拆分合并步骤，补充零件核对与质量检查环节。"
        }
    },
    {
        "instruction": "修复以下错误json：{\"task\": \"调试代码\" \"steps\": [报错信息: 定位错误, 修改代码段]}",
        "output": {
            "task": "修复代码错误",
            "steps": ["分析报错日志", "定位问题代码并修正", "测试功能完整性"],
            "fix_explanation": "转换对象为数组，规范步骤描述并补充测试验证环节。"
        }
    },
    # === 类型 4：错误修复 ===
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"去浴室洗手\" \"steps\": [\"去浴室\", \"洗\"]}",
        "output": '{"task": "前往浴室清洗双手", "steps": ["导航至浴室", "用肥皂洗手"], "fix_explanation": "修复了字段间缺失的逗号，规范了任务描述，并提升了步骤描述的具体性。"}'
    },
        {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"去浴室洗手\" \"steps\": [\"去浴室\", \"洗\"]}",
        "output": {
            "task": "前往浴室清洗双手",
            "steps": ["导航至浴室", "用肥皂洗手"],
            "fix_explanation": "修复了字段间缺失的逗号，规范了任务描述，并提升了步骤描述的具体性。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"整理书桌\" \"steps\": [\"清理杂物\", \"摆放书籍\"]}",
        "output": {
            "task": "整理书桌并归类物品",
            "steps": ["清除桌面杂物", "按分类摆放书籍和文件"],
            "fix_explanation": "补充了缺失的逗号分隔符，细化了步骤动作，明确物品归类要求。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"tasK\": \"做饭\" \"steps\": [\"洗米\", \"煮饭\", \"炒菜\"]}",
        "output": {
            "task": "完成一餐烹饪流程",
            "steps": ["淘洗大米并加水煮饭", "准备食材并烹饪菜肴", "出锅装盘检查火源关闭"],
            "fix_explanation": "修正了键名大小写错误，合并了步骤逻辑，补充了安全操作描述。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"清洁房间\" \"steps\": [\"扫地, 拖地\"]}",
        "output": {
            "task": "全面清洁房间",
            "steps": ["使用扫帚清扫地面", "用拖把擦拭地面污渍并进行消毒"],
            "fix_explanation": "分离了合并的步骤，增加了工具说明和卫生要求，修正了逗号使用错误。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"洗衣服\" \"steps\": [1. 放衣服, 2. 启动洗衣机]}",
        "output": {
            "task": "操作洗衣机清洗衣物",
            "steps": ["将衣物分类放入洗衣机", "添加洗涤剂并启动程序"],
            "fix_explanation": "修正了步骤编号格式，补充了衣物分类和洗涤剂细节，改为文本描述。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"购物\" \"steps\": [\"去超市\", \"结账离开\"]}",
        "output": {
            "task": "完成超市采购任务",
            "steps": ["抵达超市并选取商品", "前往收银台结账并核对清单"],
            "fix_explanation": "添加了选取商品环节，明确了结账流程，修正了步骤衔接缺失的问题。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"锻炼\" \"steps\": [\"跑步\", \"健身\"]}",
        "output": {
            "task": "执行每日健身计划",
            "steps": ["进行户外跑步热身", "完成力量训练及拉伸放松"],
            "fix_explanation": "细化了运动类型及阶段，补充了热身和放松环节，避免步骤模糊。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"学习\" \"steps\": [\"看书\", \"做笔记\", \"复习知识点\" \"背诵\" ]}",
        "output": {
            "task": "系统性完成学习任务",
            "steps": ["阅读教材并标记重点", "整理思维导图记录笔记", "复习知识点并背诵关键内容"],
            "fix_explanation": "修正了末尾缺失的逗号，将背诵融入复习环节，结构化了学习流程。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"准备会议\" \"steps\": [\"收集资料\", \"写报告\", \"打印文件\"]}",
        "output": {
            "task": "完成会议筹备工作",
            "steps": ["收集并整理会议所需资料", "撰写会议报告并审核内容", "打印文件并检查设备状态"],
            "fix_explanation": "补充了资料整理和审核环节，增加了设备检查步骤确保会议顺畅。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"浇花\" \"steps\": [\"取水\", \"浇水\", \"晒太阳\"]}",
        "output": {
            "task": "正确养护植物",
            "steps": ["取水并确认水温适宜", "沿植物根部缓慢浇水", "将植物移至散射光处避免暴晒"],
            "fix_explanation": "添加了水温控制要求，修正了浇水方式，调整了晒太阳步骤至合理位置。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"组装家具\" \"steps\": [\"阅读说明书\", \"安装零件\"]}",
        "output": {
            "task": "独立完成家具组装",
            "steps": ["仔细阅读说明书并核对零件清单", "按步骤安装零件并检查稳固性"],
            "fix_explanation": "补充了核对零件环节，强调了安装后的稳固性检查，避免遗漏关键步骤。"
        }
    },
    {
        "instruction": "修复以下错误JSON并增强描述：{\"task\": \"调试代码\" \"steps\": [\"报错\", \"修改\", \"测试\"]}",
        "output": {
            "task": "修复程序错误并验证功能",
            "steps": ["定位并分析报错信息", "修改代码逻辑并验证语法正确性", "运行测试用例确保功能正常"],
            "fix_explanation": "将抽象步骤转化为具体操作，补充了分析、验证和测试环节，提升技术准确性。"
        }
    }
]

print(f"✅ 共构造 {len(sft_training_data)} 条训练数据")
assert len(sft_training_data) >= 50, f"❌ 数据量不足！需要 \u2265 50 条，当前只有 {len(sft_training_data)} 条"

✅ 共构造 50 条训练数据


---
## 🧾 Step 2：SFT 训练前测试（基线记录）

在训练之前，先记录模型在测试题上的表现，用于后续对比。

In [19]:
test_questions = [
    {
        "id": "A",
        "prompt": "去卧室拿书然后去阳台晒太阳。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
        "description": "类型1·结构化任务规划 → reasoning + plan"
    },
    {
        "id": "B",
        "prompt": "用户要从超市生鲜区取一袋牛奶，到自助收银机结账，再把牛奶放进背包离开卖场。请输出 JSON，含 reasoning（简要说明顺序依据）与 plan（数组，每项含 step、action、target，必要时含 purpose）。",
        "description": "类型1·结构化任务规划（多地点）→ reasoning + plan"
    },
    {
        "id": "C",
        "prompt": "场景：停电后要先取手电筒、检查配电箱是否跳闸、若跳闸则复位、再逐一打开房间灯确认供电恢复。请先逐步理清因果与先后顺序，再只输出一段合法 JSON：字段 reasoning 中用若干短句写清推理链，字段 plan 为步骤数组（step/action/target/purpose 等），风格与课程示例一致。不要输出 JSON 以外的文字。",
        "description": "类型2·多步推理 → 推理链 + JSON plan"
    },
    {
        "id": "D",
        "prompt": "只输出一行合法 JSON（无 Markdown、无解释）。字段必须为：reasoning（字符串，1–2 句说明为何这样安排）、task（字符串）、plan（对象数组，每项至少含 step:int, action:str, target:str）、meta（对象，仅含 difficulty:\"low\"|\"medium\"|\"high\" 与 tags:字符串数组）。用户任务：从玄关取雨伞，去地铁站乘车上班。",
        "description": "类型3·格式严格约束 → 指定字段与 meta"
    },
    {
        "id": "E",
        "prompt": '修复以下错误 JSON 并增强描述：{"task": "去客厅看电影" "steps": ["去客厅" "看"], "device": 电视}',
        "description": "类型4·反例修复（缺失逗号与结构）"
    },
    {
        "id": "F",
        "prompt": '修复以下错误 JSON 并增强描述：{"task": "去花园浇花" "steps": ["拿水壶" "浇水"] "note": 记得关水龙头}',
        "description": "类型4·反例修复（引号与类型混用）"
    },
]

print("=" * 60)
print("📊 SFT 训练前 — 基线测试")
print("=" * 60)

before_sft_results = {}
for q in test_questions:
    print(f"\n--- 测试题 {q['id']}：{q['description']} ---")
    result = generate_response(q["prompt"])
    before_sft_results[q['id']] = result
    print()

📊 SFT 训练前 — 基线测试

--- 测试题 A：类型1·结构化任务规划 → reasoning + plan ---
```json
{
  "reasoning": "该计划旨在平衡室内阅读与室外放松，通过分时段活动避免过度疲劳，同时确保在安全、舒适的环境下进行。",
  "plan": [
    {
      "action": "前往卧室",
      "details": "携带书籍，前往卧室取书。",
      "duration": "约 5 分钟"
    },
    {
      "action": "阅读",
      "details": "在卧室中安静阅读书籍，保持专注。",
      "duration": "约 30 分钟"
    },
    {
      "action": "前往阳台",
      "details": "离开卧室，前往阳台区域。",
      "duration": "约 5 分钟"
    },
    {
      "action": "晒太阳",
      "details": "在阳台处进行短时间晒太阳，补充维生素 D 并放松身心。",
      "duration": "约 10-15 分钟"
    },
    {
      "action": "返回室内",
      "details": "结束户外活动后，返回卧室。",
      "duration": "约 5 分钟"
    }
  ]
}
```<|im_end|>


--- 测试题 B：类型1·结构化任务规划（多地点）→ reasoning + plan ---
```json
{
  "reasoning": "用户需先获取商品（取牛奶），再处理支付（结账），最后离开卖场（放背包）。因此顺序为：生鲜区取货 -> 收银机结账 -> 背包离开。",
  "plan": [
    {
      "step": 1,
      "action": "取货",
      "target": "生鲜区货架",
      "purpose": "获取牛奶商品"
    },
    {
      "step": 2,
      "action": "结账",
      "target": "

---
## 🧾 Step 3：将数据转换为训练格式并开始 SFT 训练（10 分）

**说明：** `UnslothVisionDataCollator` 与 Qwen3.5-VL 官方示例一致，要求每条 user 消息中同时包含 **text** 与 **image**。下方 `convert_sft_data` 会自动附加空白占位图；你仍只需维护 `instruction` / `output`。

In [20]:
def convert_sft_data(sample):
    conversation = [
        {"role": "user", "content": [{"type": "text", "text": sample["instruction"]}]},
        {"role": "assistant", "content": [{"type": "text", "text": sample["output"]}]},
    ]
    return {"messages": conversation}

converted_sft_dataset = [convert_sft_data(s) for s in sft_training_data]
print(f"✅ 已转换 {len(converted_sft_dataset)} 条训练数据")
print(f"\n示例数据：")
print(converted_sft_dataset[0])

✅ 已转换 50 条训练数据

示例数据：
{'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': '用户要去厨房拿水杯，然后回书房工作。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '{"reasoning": "该任务需要依次从当前位置前往厨房取回水杯，再前往书房进行工作。", "plan": [{"step": 1, "action": "navigate", "target": "厨房", "purpose": "取水杯"}, {"step": 2, "action": "pick_up", "target": "水杯"}, {"step": 3, "action": "navigate", "target": "书房", "purpose": "工作"}, {"step": 4, "action": "sit_down", "target": "书桌"}]}'}]}]}


In [21]:
import torch._dynamo
torch._dynamo.config.disable = True

from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

sft_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_sft_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "sft_outputs",
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 1024,
    ),
)

print("✅ SFT Trainer 已配置")

Unsloth: Model does not have a default image size - using 512
Unsloth: Switching to float32 training since model cannot work with float16
✅ SFT Trainer 已配置


In [22]:
print("🚀 开始 SFT 训练...")
sft_stats = sft_trainer.train()
print(f"\n✅ SFT 训练完成！用时 {sft_stats.metrics['train_runtime']:.1f} 秒")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


🚀 开始 SFT 训练...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 9 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 16,819,200 of 2,230,060,864 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,1.997046
10,1.429665
15,0.989963
20,0.709743
25,0.532353
30,0.420171
35,0.344731
40,0.269740
45,0.187949
50,0.164797



✅ SFT 训练完成！用时 328.5 秒


In [ ]:
model.save_pretrained("sft_model")
tokenizer.save_pretrained("sft_model")
print("✅ SFT 模型已保存到 sft_model/")

---
## 🧾 Step 4：SFT 训练后测试与对比（10 分）

In [23]:
print("=" * 60)
print("📊 SFT 训练后 — 测试")
print("=" * 60)

after_sft_results = {}
for q in test_questions:
    print(f"\n--- 测试题 {q['id']}：{q['description']} ---")
    result = generate_response(q["prompt"])
    after_sft_results[q['id']] = result
    print()

📊 SFT 训练后 — 测试

--- 测试题 A：类型1·结构化任务规划 → reasoning + plan ---
{'reasoning': '该任务需先获取阅读材料，再转移至休闲区域享受自然光。', 'plan': [{'step': 1, 'action': 'navigate', 'target': '卧室', 'purpose': '拿书'}, {'step': 2, 'action': 'pick_up', 'target': '书'}, {'step': 3, 'action': 'navigate', 'target': '阳台', 'purpose': '晒太阳'}, {'step': 4, 'action': 'relax', 'target': '身体'}]}<|im_end|>


--- 测试题 B：类型1·结构化任务规划（多地点）→ reasoning + plan ---
{'reasoning': '该任务需先获取生鲜商品，再执行结账流程，最后转移物品至外出携带。', 'plan': [{'step': 1, 'action': 'navigate', 'target': '生鲜区', 'purpose': '取牛奶'}, {'step': 2, 'action': 'pick_up', 'target': '牛奶', 'purpose': '结账使用'}, {'step': 3, 'action': 'checkout', 'target': '自助收银机', 'purpose': '结算商品费用'}, {'step': 4, 'action': 'carry', 'target': '背包', 'purpose': '放置牛奶'}, {'step': 5, 'action': 'exit', 'target': '卖场', 'purpose': '离开购物区域'}]}<|im_end|>


--- 测试题 C：类型2·多步推理 → 推理链 + JSON plan ---
{'reasoning': '停电后需先获取照明工具，再检查电力设施状态，确认故障后修复并验证供电恢复，确保安全与功能正常。', 'plan': [{'step': 1, 'action': 'navigate', 'target': '手电筒', 'pu

In [24]:
print("=" * 60)
print("📊 SFT 前后对比")
print("=" * 60)

for q in test_questions:
    print(f"\n{'='*50}")
    print(f"测试题 {q['id']}：{q['description']}")
    print(f"{'='*50}")
    print(f"\n🔹 训练前输出：")
    print(before_sft_results[q['id']])
    print(f"\n🔹 训练后输出：")
    print(after_sft_results[q['id']])

📊 SFT 前后对比

测试题 A：类型1·结构化任务规划 → reasoning + plan

🔹 训练前输出：
```json
{
  "reasoning": "该计划旨在平衡室内阅读与室外放松，通过分时段活动避免过度疲劳，同时确保在安全、舒适的环境下进行。",
  "plan": [
    {
      "action": "前往卧室",
      "details": "携带书籍，前往卧室取书。",
      "duration": "约 5 分钟"
    },
    {
      "action": "阅读",
      "details": "在卧室中安静阅读书籍，保持专注。",
      "duration": "约 30 分钟"
    },
    {
      "action": "前往阳台",
      "details": "离开卧室，前往阳台区域。",
      "duration": "约 5 分钟"
    },
    {
      "action": "晒太阳",
      "details": "在阳台处进行短时间晒太阳，补充维生素 D 并放松身心。",
      "duration": "约 10-15 分钟"
    },
    {
      "action": "返回室内",
      "details": "结束户外活动后，返回卧室。",
      "duration": "约 5 分钟"
    }
  ]
}
```

🔹 训练后输出：
{'reasoning': '该任务需先获取阅读材料，再转移至休闲区域享受自然光。', 'plan': [{'step': 1, 'action': 'navigate', 'target': '卧室', 'purpose': '拿书'}, {'step': 2, 'action': 'pick_up', 'target': '书'}, {'step': 3, 'action': 'navigate', 'target': '阳台', 'purpose': '晒太阳'}, {'step': 4, 'action': 'relax', 'target': '身体'}]}

测试题 B：类型1·结构化任务规划（多地点）→ reasoning + p

#### ✏️ SFT 效果分析

请在下方写出你的分析：

<!-- ✏️ 学生填写 -->

**SFT 训练前后对比分析：**

1. **格式稳定性方面的变化：**
   - 格式稳定性显著下降，训练后输出不符合 JSON 规范，需进一步修正。

2. **推理质量方面的变化：**
   - 推理质量在简洁性上有所提升，但存在信息丢失、步骤泛化过度和安全性问题，需平衡简洁性与完整性。

3. **主要改善和仍存在的问题：**
   - 主要问题集中在格式合法性和信息完整性上，需优化模型以确保输出既符合规范，又保留关键细节和逻辑。

---
---

# ❤️ Part 3 — DPO 偏好学习（35 分）

## 🎯 目标

通过偏好学习，让模型学会：
- 更符合人类偏好的输出
- 更好的 reasoning 风格
- 避免冗余或错误结构

**🔑 重要：** DPO 将在**原始模型**上进行训练（而非 SFT 后的模型），以便对比 SFT 和 DPO 两种方法的独立效果。


---
## 🧾 Step 1：构造偏好数据（15 分）

每条数据包含一个 `prompt`、一个 `chosen`（优质输出）和一个 `rejected`（劣质输出）。

### 📌 数据设计要求

必须体现**细粒度偏好差异**：

| 维度 | 优质输出（chosen） | 劣质输出（rejected） |
|------|-------------|----------------|
| 格式 | JSON 格式正确、字段完整 | 格式错误或字段缺失 |
| 推理说明 | 简洁、客观、切题 | 冗长、废话多、跑题 |
| 行动计划 | 步骤完整、逻辑清晰 | 步骤缺失或顺序不合理 |
| 修复说明 | 清晰具体 | 模糊或遗漏 |

### 示例

```python
{
    "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
    "chosen": '{"reasoning": "该任务需要从当前位置依次导航至厨房取水，再前往客厅休息。", "plan": [{"step": 1, "action": "navigate", "target": "厨房", "purpose": "取水"}, {"step": 2, "action": "pick_up", "target": "水"}, {"step": 3, "action": "navigate", "target": "客厅", "purpose": "休息"}, {"step": 4, "action": "sit_down", "target": "沙发"}]}',
    "rejected": '{"reasoning": "用户想要去厨房。厨房是可以找到水的地方。水对身体很重要。喝水有利于健康。拿了水之后用户要去客厅。客厅通常是一个舒适的地方。", "plan": [{"step": 1, "action": "go", "target": "厨房"}]}'
}
```

请构造 **≥ 30 条**偏好数据。

In [32]:
# ✏️ 学生填写：构造你的偏好数据（≥ 30 条）

dpo_training_data = [
    {
        "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
        "chosen": '{"reasoning": "该任务需要从当前位置依次导航至厨房取水，再前往客厅休息。", "plan": [{"step": 1, "action": "navigate", "target": "厨房", "purpose": "取水"}, {"step": 2, "action": "pick_up", "target": "水"}, {"step": 3, "action": "navigate", "target": "客厅", "purpose": "休息"}, {"step": 4, "action": "sit_down", "target": "沙发"}]}',
        "rejected": '{"reasoning": "用户想要去厨房。厨房是可以找到水的地方。水对身体很重要。喝水有利于健康。拿了水之后用户要去客厅。客厅通常是一个舒适的地方。", "plan": [{"step": 1, "action": "go", "target": "厨房"}]}'
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "用户需先导航至厨房取水，再前往客厅休息，步骤需包含取水动作及空间转移的连贯性。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房", "purpose": "取水" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅", "purpose": "休息" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划遗漏取水动作，仅导航至厨房后直接休息，逻辑不完整。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "navigate", "target": "客厅" },
          { "step": 3, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "任务需明确空间转移顺序：厨房→取水→客厅→休息，动作与目标需准确对应。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划中取水后导航至卧室而非客厅，与用户指令矛盾。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "卧室" },
          { "step": 4, "action": "sit_down", "target": "床" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "用户需求包含两个关键动作：获取物品（水）和休息，计划需按顺序完成。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "动作使用错误，'go' 和 'sit' 与标准动作定义不符（应为 navigate 和 sit_down）。",
        "plan": [
          { "step": 1, "action": "go", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "go", "target": "客厅" },
          { "step": 4, "action": "sit", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "需先完成取水任务，再导航至休息区域，确保步骤顺序正确且无冗余动作。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划添加了不必要的饮水步骤，超出用户需求（用户未要求当场喝水）。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "drink", "target": "水" },
          { "step": 4, "action": "navigate", "target": "客厅" },
          { "step": 5, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "任务需分解为两个阶段：获取资源（水）和转移至休息区，动作和目标需明确对应。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "步骤顺序错误，先导航至客厅再返回厨房取水，违背用户指令顺序。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "客厅" },
          { "step": 2, "action": "navigate", "target": "厨房" },
          { "step": 3, "action": "pick_up", "target": "水" },
          { "step": 4, "action": "navigate", "target": "客厅" },
          { "step": 5, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "用户目标明确，需先取水再休息，计划需包含完整步骤链且无遗漏动作。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划中取水动作的目标错误，应为 '水' 而非 '杯子'，可能导致执行偏差。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "杯子" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "需确保空间转移与物品获取的连贯性：厨房（取水）→ 客厅（休息），动作准确且步骤完整。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划缺少最终休息动作，仅完成取水与导航，导致任务未闭环。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "用户需求包含两个地点和两个动作，计划需按顺序执行且不遗漏关键步骤。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划中重复导航至厨房，步骤冗余且不符合用户指令的单一流程要求。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "navigate", "target": "厨房" },
          { "step": 5, "action": "navigate", "target": "客厅" },
          { "step": 6, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "任务流程需先导航至厨房执行取水，随后导航至客厅执行休息动作，确保步骤目的明确。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房", "purpose": "取水" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅", "purpose": "休息" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划中缺少 purpose 字段，导致步骤意图不明确，易引发执行歧义。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "需先完成取水前提条件，再执行休息动作，步骤顺序不可颠倒，且动作必须完整。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划中遗漏第二步取水动作，仅导航至厨房后直接休息，任务目标未完成。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "navigate", "target": "客厅" },
          { "step": 3, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "用户需先导航至厨房取水，携带水前往客厅，最后在沙发休息。计划需包含完整路径和动作。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房", "purpose": "取水" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅", "purpose": "休息" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划遗漏了从厨房到客厅的导航步骤，导致逻辑不连贯。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "任务核心是获取水后前往客厅休息，需明确动作对象和空间转移顺序。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "取水动作的目标错误，应指向'水'而非'杯子'，可能引发执行错误。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "杯子" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "需按顺序完成两个子任务：厨房取水（动作：pick_up）和客厅休息（动作：sit_down）。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划中添加了无关动作（打开电视），偏离用户核心需求（休息）。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "turn_on", "target": "电视" },
          { "step": 5, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "用户需先到达厨房获取水，再转移至客厅进行休息，步骤顺序不可颠倒。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "步骤顺序错误，先导航至客厅再取水，与用户指令逻辑矛盾。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "客厅" },
          { "step": 2, "action": "navigate", "target": "厨房" },
          { "step": 3, "action": "pick_up", "target": "水" },
          { "step": 4, "action": "navigate", "target": "客厅" },
          { "step": 5, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "计划需明确取水动作（pick_up）和休息动作（sit_down），并确保空间转移正确。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "动作名称错误，使用'sit'而非'sit_down'，不符合标准动作定义。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "需确保计划包含所有必要步骤：导航至厨房、取水、导航至客厅、坐下休息。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "遗漏取水动作，仅导航至厨房后直接前往客厅，任务未完整执行。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "navigate", "target": "客厅" },
          { "step": 3, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "用户需求涉及两个地点和两个动作，计划需按顺序连接：厨房（取水）→ 客厅（休息）。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "导航目标错误，将客厅误写为'卧室'，偏离用户指令。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "卧室" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "任务需分解为四个关键步骤：导航、取水、导航、休息，动作与目标需精确匹配。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "添加了冗余步骤（检查水是否足够），超出用户需求范围。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "check", "target": "水量是否足够" },
          { "step": 4, "action": "navigate", "target": "客厅" },
          { "step": 5, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "用户需先完成取水前置任务，再执行休息动作，计划步骤需连贯且无遗漏。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划中将'pick_up'错误地替换为'drink'，导致逻辑矛盾（用户未要求现场饮水）。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "drink", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "需确保空间转移和动作顺序正确：厨房→取水→客厅→坐下，步骤编号连续且无重复。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "步骤编号重复（step 3 出现两次），导致计划结构混乱且不可执行。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 3, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "用户需先导航至厨房取水，携带水前往客厅休息。计划需包含完整路径和动作，无冗余步骤。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房", "purpose": "取水" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅", "purpose": "休息" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "计划中包含重复的导航步骤，浪费资源且不符合效率要求。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "navigate", "target": "厨房" },
          { "step": 5, "action": "navigate", "target": "客厅" },
          { "step": 6, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "取水动作必须在厨房执行，休息在客厅，动作与位置需严格对应。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "动作与位置不匹配，在客厅执行取水动作，违背用户指令逻辑。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "客厅" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "厨房" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "每个动作必须明确目标对象，避免歧义或执行错误。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "pick_up 动作缺少目标，导致执行时无法确定操作对象。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "计划需简洁，仅包含必要步骤，避免无关动作干扰核心任务。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "添加无关的确认步骤，增加复杂性且无实际必要性。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "confirm", "target": "水已取" },
          { "step": 4, "action": "navigate", "target": "客厅" },
          { "step": 5, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "任务顺序不可颠倒，必须优先完成取水，再休息。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "步骤顺序颠倒，休息动作在取水前，导致任务无法完成。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "客厅" },
          { "step": 2, "action": "sit_down", "target": "沙发" },
          { "step": 3, "action": "navigate", "target": "厨房" },
          { "step": 4, "action": "pick_up", "target": "水" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "所有导航动作必须指定目标位置，确保路径明确。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "导航动作缺少 target 参数，导致路径规划失败。",
        "plan": [
          { "step": 1, "action": "navigate" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "计划需符合用户指令的因果关系：取水后休息，无逻辑矛盾。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "逻辑矛盾，导航至卧室取水，与用户指定厨房位置冲突。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "卧室" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "动作类型需准确，使用标准动词（pick_up 而非 grab）确保兼容性。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "使用非标准动作动词（grab），可能导致执行系统无法识别。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "grab", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      }
    },
    {
      "prompt": "用户要去厨房拿水，然后去客厅休息。请输出 JSON 格式的行动计划，包含 reasoning 和 plan。",
      "chosen": {
        "reasoning": "每个步骤需唯一且有效，避免重复动作导致冗余执行。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" }
        ]
      },
      "rejected": {
        "reasoning": "重复执行坐下动作，不符合任务要求且增加不必要的步骤。",
        "plan": [
          { "step": 1, "action": "navigate", "target": "厨房" },
          { "step": 2, "action": "pick_up", "target": "水" },
          { "step": 3, "action": "navigate", "target": "客厅" },
          { "step": 4, "action": "sit_down", "target": "沙发" },
          { "step": 5, "action": "sit_down", "target": "沙发" }
        ]
      }
    }
]

print(f"✅ 共构造 {len(dpo_training_data)} 条偏好数据")
assert len(dpo_training_data) >= 30, f"❌ 数据量不足！需要 ≥ 30 条，当前只有 {len(dpo_training_data)} 条"

✅ 共构造 30 条偏好数据


---
## 🧾 Step 2：重新加载原始模型用于 DPO 训练

**重要：** 我们先释放 SFT 模型的显存，然后重新加载原始 Qwen3.5-2B 模型并挂载全新的 LoRA 适配器，确保 DPO 是在原始模型上训练的。

In [33]:
import gc

del model
del sft_trainer
gc.collect()
torch.cuda.empty_cache()
print("✅ 已释放 SFT 模型显存")

print("\n🔄 重新加载原始模型...")
model, tokenizer = load_model_with_lora()
print("✅ 原始模型已重新加载（全新 LoRA 适配器）")

✅ 已释放 SFT 模型显存

🔄 重新加载原始模型...
==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Unsloth: Making `model.base_model.model.model.language_model` require gradients
✅ 原始模型已重新加载（全新 LoRA 适配器）


In [34]:
print("=" * 60)
print("📊 DPO 训练前（原始模型）— 基线测试")
print("=" * 60)

before_dpo_results = {}
for q in test_questions:
    print(f"\n--- 测试题 {q['id']}：{q['description']} ---")
    result = generate_response(q["prompt"])
    before_dpo_results[q['id']] = result
    print()

📊 DPO 训练前（原始模型）— 基线测试

--- 测试题 A：类型1·结构化任务规划 → reasoning + plan ---
```json
{
  "reasoning": "该计划旨在平衡室内阅读与室外放松，通过分时段活动避免过度疲劳，同时确保在安全、舒适的环境下进行。",
  "plan": [
    {
      "action": "前往卧室",
      "details": "携带书籍，前往卧室取书。",
      "duration": "约 5 分钟"
    },
    {
      "action": "阅读",
      "details": "在卧室中安静阅读书籍，保持专注。",
      "duration": "约 30 分钟"
    },
    {
      "action": "前往阳台",
      "details": "离开卧室，前往阳台区域。",
      "duration": "约 5 分钟"
    },
    {
      "action": "晒太阳",
      "details": "在阳台处进行短时间晒太阳，补充维生素 D 并放松身心。",
      "duration": "约 10-15 分钟"
    },
    {
      "action": "返回室内",
      "details": "结束户外活动后，返回卧室。",
      "duration": "约 5 分钟"
    }
  ]
}
```<|im_end|>


--- 测试题 B：类型1·结构化任务规划（多地点）→ reasoning + plan ---
```json
{
  "reasoning": "用户需先获取商品（取牛奶），再处理支付（结账），最后离开卖场（放背包）。因此顺序为：生鲜区取货 -> 收银机结账 -> 背包离开。",
  "plan": [
    {
      "step": 1,
      "action": "取货",
      "target": "生鲜区货架",
      "purpose": "获取牛奶商品"
    },
    {
      "step": 2,
      "action": "结账",
      "targe

---
## 🧾 Step 3：DPO 训练（10 分）

In [35]:
from datasets import Dataset

def format_dpo_data(data_list):
    """纯文本 DPO 格式（本作业不涉及真实图像，绕过 Vision DPO 的图像处理问题）。"""
    formatted = []
    for item in data_list:
        formatted.append({
            "prompt": [
                {"role": "user", "content": item["prompt"]}
            ],
            "chosen": [
                {"role": "assistant", "content": item["chosen"]}
            ],
            "rejected": [
                {"role": "assistant", "content": item["rejected"]}
            ],
        })
    return Dataset.from_list(formatted)

dpo_dataset = format_dpo_data(dpo_training_data)
print(f"✅ DPO 数据集已准备：{len(dpo_dataset)} 条")
print(f"\n数据集字段：{dpo_dataset.column_names}")

ArrowInvalid: cannot mix struct and non-struct, non-null values

In [ ]:
import torch._dynamo
torch._dynamo.config.disable = True

from trl import DPOTrainer, DPOConfig
from transformers.models.auto.modeling_auto import MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES

FastVisionModel.for_training(model)

_vision_keys_backup = {}
for k in list(MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES.keys()):
    if "qwen" in k.lower():
        _vision_keys_backup[k] = MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES.pop(k)

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    train_dataset = dpo_dataset,
    processing_class = tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 40,
        learning_rate = 5e-5,
        logging_steps = 5,
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = "dpo_outputs",
        report_to = "none",
        remove_unused_columns = False,
        max_length = 1024,
        max_prompt_length = 512,
    ),
)

MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES.update(_vision_keys_backup)

print("✅ DPO Trainer 已配置")

In [ ]:
print("🚀 开始 DPO 训练...")
dpo_stats = dpo_trainer.train()
print(f"\n✅ DPO 训练完成！用时 {dpo_stats.metrics['train_runtime']:.1f} 秒")

In [ ]:
model.save_pretrained("dpo_model")
tokenizer.save_pretrained("dpo_model")
print("✅ DPO 模型已保存到 dpo_model/")

---
## 🧾 Step 4：效果验证与三方对比（10 分）

In [ ]:
print("=" * 60)
print("📊 DPO 训练后 — 测试")
print("=" * 60)

after_dpo_results = {}
for q in test_questions:
    print(f"\n--- 测试题 {q['id']}：{q['description']} ---")
    result = generate_response(q["prompt"])
    after_dpo_results[q['id']] = result
    print()

In [ ]:
import json

def evaluate_output(output_text):
    score = 0
    details = []

    try:
        parsed = json.loads(output_text.strip())
        score += 3
        details.append("✅ JSON 格式合法 (+3)")
    except:
        details.append("❌ JSON 格式不合法 (+0)")
        return score, details

    if "reasoning" in parsed:
        r = parsed["reasoning"]
        if 20 < len(r) < 300:
            score += 2
            details.append("✅ reasoning 长度合理 (+2)")
        else:
            score += 1
            details.append("⚠️ reasoning 长度不够理想 (+1)")
    else:
        details.append("❌ 缺少 reasoning 字段 (+0)")

    if "plan" in parsed:
        p = parsed["plan"]
        if isinstance(p, list) and len(p) >= 2:
            has_structure = all(
                isinstance(s, dict) and "step" in s and "action" in s
                for s in p
            )
            if has_structure:
                score += 3
                details.append("✅ plan 结构完整 (+3)")
            else:
                score += 1
                details.append("⚠️ plan 内部结构不标准 (+1)")
        else:
            score += 1
            details.append("⚠️ plan 步骤数不足 (+1)")
    elif "steps" in parsed:
        score += 2
        details.append("✅ 包含 steps 字段 (+2)")
    elif "fix_explanation" in parsed:
        score += 2
        details.append("✅ 包含 fix_explanation 字段 (+2)")

    return score, details

print("=" * 70)
print("📊 三方对比总结：原始模型 vs SFT vs DPO")
print("=" * 70)

stages = [
    ("原始模型（基线）", before_sft_results),
    ("SFT 训练后", after_sft_results),
    ("DPO 训练后", after_dpo_results),
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"测试题 {q['id']}：{q['description']}")
    print(f"{'='*60}")
    for stage_name, results in stages:
        score, details = evaluate_output(results[q['id']])
        print(f"\n  📌 {stage_name}：得分 {score}/8")
        for d in details:
            print(f"     {d}")

print("\n" + "=" * 70)
print("总分汇总：")
for stage_name, results in stages:
    total = sum(evaluate_output(results[q['id']])[0] for q in test_questions)
    print(f"  {stage_name}：{total} / {8 * len(test_questions)}")

#### ✏️ 最终分析

请在下方写出你的全面分析：

<!-- ✏️ 学生填写 -->

**三方对比分析（原始模型 vs SFT vs DPO）：**

1. **Prompt Engineering vs SFT：**
   - Prompt Engineering 的局限性是什么？
   - SFT 带来了哪些改善？
   - （请填写）

2. **SFT vs DPO（两者都从原始模型训练）：**
   - 哪种方法在哪些方面更有优势？
   - 它们各自的特点是什么？
   - （请填写）

3. **输出稳定性、reasoning 质量、格式一致性的变化趋势：**
   - （请填写）

4. **如果要进一步改善模型表现，你会怎么做？**
   - （请填写）

---
## 📋 作业提交清单

请确认你已完成以下所有内容：

| 部分 | 内容 | 完成 |
|------|------|------|
| Part 1 | 三个任务的 Prompt 设计与输出 | ☐ |
| Part 2 | SFT 训练数据 | ☐ |
| Part 2 | SFT 训练前后对比分析 | ☐ |
| Part 3 | DPO 偏好数据 | ☐ |
| Part 3 | 三方对比分析（原始 vs SFT vs DPO） | ☐ |

**提交方式：** 下载此 notebook（`.ipynb` 文件）并上传到课程平台。